## Workflow vs Agent vs Multi-Agent

- 단순하고 반복적인 작업은 **Workflow**로 안정성을 잡고, 
- 복잡하고 추론이 필요한 구간은 **Agent**로 유연성을 더하세요. 
- 그리고 그 에이전트가 감당하기에 일이 너무 크다면 **Multi-Agent** 팀을 꾸리면 됩니다!

### 1. 기본 개념 정의
* **Workflow (결정론적 구조):** 모든 실행 경로와 조건 분기를 개발자가 미리 코드로 설계한 '지도'입니다. 정해진 선(Edge)을 따라서만 움직입니다.
* **Agent (자율적 구조):** LLM에게 **목표 Goal**와 **도구 Tool**를 주고, "어떤 도구를 어떤 순서로 쓸지" 결정을 LLM에게 맡긴 '지능형 실행체'입니다.


### 2. Workflow vs Agent: 핵심 차이점

| 비교 항목 | Workflow (정적) | Agent (동적) |
| --- | --- | --- |
| **예외 상황 유연성** | 데이터가 없거나 에러 발생 시 미리 정의된 분기로만 대응 가능 | "정보가 부족하네? 다른 도구를 써볼까?"라고 스스로 판단하여 복구 |
| **경로 최적화** | 질문의 난이도와 상관없이 모든 노드를 순차적으로 통과 | 쉬운 질문은 바로 답변, 어려운 질문은 도구를 여러 번 사용하여 최적화 |
| **비판적 사고** | 앞 단계의 결과를 무비판적으로 수용하여 다음 단계로 전달 | 결과가 만족스럽지 않으면 스스로 루프를 돌며 다시 시도(Self-Correction) |
| **제어권** | **개발자** (강력한 통제, 높은 예측 가능성) | **LLM** (높은 유연성, 예측 불가능성 존재) |



### 3. Workflow를 Agent로 리팩토링하기
* **노드의 단순화:** 수십 개의 `if/else` 노드 대신, **[Agent Node]** 와 **[Tool Node]** 위주의 심플한 구조로 압축됩니다.
* **기법의 도구화:** `Self-RAG`나 `Corrective-RAG` 같은 복잡한 로직이 에이전트가 필요할 때 꺼내 쓰는 **'검증 도구'** 나 **'검색 도구'** 의 형태로 변합니다.
* **상태 관리:** 재시도 제한 등은 그래프를 꼬는 대신, `State`의 카운터 변수나 시스템 프롬프트로 훨씬 깔끔하게 제어합니다.



### 4. 단일 에이전트 vs 멀티 에이전트

* **단일 에이전트 (Single Agent):** 한 명의 LLM이 모든 도구를 다 가집니다. 구현은 쉽지만, 도구가 너무 많아지면 주의력이 분산되고 분석의 깊이가 얕아질 수 있습니다.
* **멀티 에이전트 (Multi-Agent):** 목표를 쪼개어 전문 분야별 에이전트(시장 조사, 기업 분석 등)에게 배분합니다. 각 에이전트가 자기 역할에만 집중하므로 정확도가 높고 결과물이 정교해집니다.
    * **Supervisor (관리자 방식):** 팀장 에이전트가 전체 그림을 보고 하위 에이전트들에게 일을 배분하고 결과를 검수합니다. **품질 관리와 절차**가 중요한 비즈니스 로직에 유리합니다.
    * **Network (피어 방식):** 에이전트들이 수평적으로 소통하며 자유롭게 바톤 터치를 합니다. **창의적인 협업이나 복잡한 상호작용**이 필요한 연구/기획 작업에 유리합니다.

## LangGraph `create_agent`
### 1. create_agent "자동 완성 패키지"
- from langchain.agents import create_agent
- 수동으로 했던 번거로운 작업들을 내부적으로 미리 다 처리해 줌
    - LLM + Tool 바인딩: llm.bind_tools(tools)를 알아서 수행합니다.
    - 시스템 프롬프트 설정: state_modifier에 넣은 문구를 시스템 메시지로 고정해 줍니다.
    - ToolNode 연결: 에이전트가 도구를 쓰고 싶을 때 바로 실행할 수 있도록 내부적인 루프(Loop) 구조를 이미 갖추고 있습니다.
- 즉, market_research_agent = create_agent(...)를 실행하는 순간, "생각하고(LLM) + 도구 쓰고(ToolNode) + 다시 생각하는" 일련의 작은 그래프가 이미 완성되어 리턴되는 것이라고 이해하시면 돼요.

### 2. 왜 이전 방식과 다른가요?
- 이전 방식(수동 연결)
    - 노드와 엣지를 직접 하나하나 연결
    - 에이전트의 내부 로직(도구 실행 순서 등)을 내 맘대로 개조 가능
- `create_agent` 
    - 검증된 표준 에이전트 구조를 한 줄로 생성
    - 코드가 훨씬 짧아지고, 멀티 에이전트의 부품(Worker)으로 쓰기 매우 편리함
- 하지만, 에이전트를 create_agent로 만들었든 직접 짰든, 랭그래프 시스템 안으로 들어오는 순간 **"State를 받아서, 변경분만 리턴한다"** 는 공통의 룰을 따르는 것은 동일하다 

### 3. 멀티 에이전트에서의 위력
- 전문가 4명을 한 줄씩 뚝딱 만든다.
- 그들을 각각의 노드(Node) 함수 안에서 호출(invoke)한다.
- 전체 흐름만 Supervisor가 관리하게 한다.


## 주식 투자 멀티 에이전트 개발: supervisor 패턴
- (1) 시장 조사 에이전트 
- (2) 주식 조사 에이전트
- (3) 회사 조사 에이전트
- (4) 분석가 에이전트 

In [1]:
#uv add yfinance

In [2]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

llm = ChatOpenAI(model='gpt-4o')
small_llm = ChatOpenAI(model='gpt-4o-mini')

## 시장조사 에이전트
### Tool 1: Polygon API - 주식 시장의 '실시간 & 과거 수치 데이터'
- 성격: 매우 객관적이고 정확한 수치 데이터 중심입니다.
- 할 수 있는 일: 특정 종목의 현재 가격, 일일 주가 변동 리스트, 기업의 재무 지표, 티커(Ticker) 심볼 검색 등을 수행합니다.
- 왜 쓰나요? 에이전트가 "요즘 삼성전자 얼마야?" 혹은 "반도체 섹터에서 오늘 가장 많이 오른 주식이 뭐야?"라는 질문에 정확한 숫자로 대답하기 위해 사용합니다.

### Tool 2: YahooFinanceNewsTool - '최신 시장 뉴스'
- 성격: 시장의 흐름, 이슈, 전문가들의 의견 등 텍스트 중심의 정보를 가져옵니다.
- 할 수 있는 일: 특정 기업이나 시장 전체에 대한 최신 뉴스 헤드라인과 내용을 검색합니다.
- 왜 쓰나요? 주가는 숫자만으로 움직이지 않죠? "금리 인상 발표에 시장이 어떻게 반응하고 있어?" 같은 맥락과 뉴스를 파악하기 위해 사용합니다.

In [3]:
from langgraph.graph import MessagesState
from langgraph.types import Command
from typing import Literal
from langchain_core.messages import HumanMessage
from langchain_community.tools.yahoo_finance_news import YahooFinanceNewsTool

from langchain.agents import create_agent
from langchain_community.agent_toolkits.polygon.toolkit import PolygonToolkit
from langchain_community.utilities.polygon import PolygonAPIWrapper

# Polygon API를 초기화합니다.
polygon = PolygonAPIWrapper()

# Polygon API로부터 도구 모음을 생성합니다.
toolkit = PolygonToolkit.from_polygon_api_wrapper(polygon)

# 도구들을 가져옵니다.
polygon_tools = toolkit.get_tools()

# 시장 조사 도구 목록을 생성합니다: 주식 시장 뉴스 데이터 + 수치 데이터 
market_research_tools = [YahooFinanceNewsTool()] + polygon_tools

# 시장 조사 에이전트를 생성합니다.
market_research_agent = create_agent(
    model="gpt-4o-mini",
    tools=market_research_tools,
    system_prompt="You are a market researcher. Provide fact only not opinions."
)


# 상태 업데이트와 이동 제어를 동시에 하기 위해 리턴값으로 Command 객체를 사용
def market_research_node(state: MessagesState) -> Command[Literal["supervisor"]]:
    """
    시장 조사 node입니다. 주어진 state를 기반으로 시장 조사 에이전트를 호출하고,
    결과를 supervisor node로 전달합니다.

    Args:
        state (MessagesState): 현재 메시지 상태를 나타내는 객체입니다.

    Returns:
        Command: supervisor node로 이동하기 위한 명령을 반환합니다.
    """
    # 시장 조사 에이전트를 호출하여 결과를 얻습니다.
    result = market_research_agent.invoke(state)
    
    # 결과 메시지를 업데이트하고 supervisor node로 이동합니다.
    return Command(
        update={'messages': [HumanMessage(content=result['messages'][-1].content, name='market_research')]},
        goto='supervisor'
    )

USER_AGENT environment variable not set, consider setting it to identify your requests.


## 주식 조사 Agent
### 1. 사용된 도구: `get_stock_price`
- 이 도구는 내부적으로 `yfinance` 라이브러리를 사용하여 주식의 수치 데이터를 가져옵니다.
- 입력(Input): 주식 티커(미국주식 종목의 종목코드, 예: 'AAPL', 'TSLA', 'NVDA')를 받습니다.
- 기능: 해당 종목의 최근 1개월(period='1mo') 동안의 가격 데이터를 다운로드합니다.
- 출력(Output): 다운로드한 데이터를 파이썬의 dict 형태로 변환하여 에이전트에게 전달합니다.

### 2. 시장 조사 도구와의 결정적 차이
- 시장 조사 도구: "뉴스"나 "전반적인 지표"를 훑는 용도였습니다.
- 주식 조사 도구 (get_stock_price): 특정 종목을 콕 집어서 그 종목의 지난 한 달간의 구체적인 가격 추이를 확인하는 용도입니다.
- 에이전트는 이 데이터를 보고 "지난 한 달간 주가가 우상향했는지", "변동성이 심했는지" 등을 수치적으로 판단할 수 있게 됩니다.

### 3. 코딩 포인트: @tool 데코레이터
- 함수의 이름(get_stock_price)이 도구의 이름이 됩니다.
- 함수의 주석(Docstring: "Given a stock ticker...")이 에이전트에게 보내는 설명서가 됩니다. LLM은 이 설명을 읽고 "아, 주가 데이터가 필요할 땐 이 함수를 호출하면 되겠구나!"라고 이해합니다.

In [ ]:
import yfinance as yf

from langchain.tools import tool

# 커스텀 도구 정의 
@tool
def get_stock_price(ticker: str) -> dict:
    """Given a stock ticker, return the price data for the past month"""
    stock_info = yf.download(ticker, period='1mo').to_dict()
    return stock_info


stock_research_tools = [get_stock_price]
stock_research_agent = create_agent(
    model="gpt-4o-mini",
    tools=stock_research_tools,
    system_prompt="You are a stock researcher. Provide facts only not opinions"
)

def stock_research_node(state: MessagesState) -> Command[Literal["supervisor"]]:
    """
    주식 조사 node입니다. 주어진 State를 기반으로 주식 조사 에이전트를 호출하고,
    결과를 supervisor node로 전달합니다.

    Args:
        state (MessagesState): 현재 메시지 상태를 나타내는 객체입니다.

    Returns:
        Command: supervisor node로 이동하기 위한 명령을 반환합니다.
    """
    result = stock_research_agent.invoke(state)

    return Command(
        update={'messages': [HumanMessage(content=result['messages'][-1].content, name='stock_research')]},
        goto='supervisor'
    )

In [ ]:
@tool
def company_research_tool(ticker: str) -> dict:
    """Given a ticker, return the financial information and SEC filings"""
    company_info = yf.Ticker(ticker)
    financial_info = company_info.get_financials()
    sec_filings = company_info.get_sec_filings()
    return {
        'financial_info': financial_info,
        'sec_filings': sec_filings
    }

company_research_tools = [company_research_tool]
company_research_agent = create_agent(
    model="gpt-4o-mini",
    tools=company_research_tools,
    system_prompt="You are a company researcher. Provide facts only not opinions"
)

def company_research_node(state: MessagesState) -> Command[Literal["supervisor"]]:
    """
    회사 조사 node입니다. 주어진 State를 기반으로 회사 조사 에이전트를 호출하고,
    결과를 supervisor node로 전달합니다.

    Args:
        state (MessagesState): 현재 메시지 상태를 나타내는 객체입니다.

    Returns:
        Command: supervisor node로 이동하기 위한 명령을 반환합니다.
    """
    result = company_research_agent.invoke(state)

    return Command(
        update={'messages': [HumanMessage(content=result['messages'][-1].content, name='company_research')]},
        goto='supervisor'
    )

In [ ]:
from langchain_core.prompts import PromptTemplate

analyst_prompt = PromptTemplate.from_template(
    """You are a stock market analyst. Given the following information, 
Please decide wheter to buy, sell, or hold the stock.

Information:
{messages}"""
)

analyst_chain = analyst_prompt | llm

def analyst_node(state: MessagesState):
    """
    분석가 node입니다. 주어진 State를 기반으로 분석가 체인을 호출하고,
    결과 메시지를 반환합니다.

    Args:
        state (MessagesState): 현재 메시지 상태를 나타내는 객체입니다.

    Returns:
        dict: 분석 결과 메시지를 포함하는 딕셔너리를 반환합니다.
    """
    result = analyst_chain.invoke({'messages': state['messages'][1:]})

    return {'messages': [result]}

In [ ]:
from typing import Literal
from typing_extensions import TypedDict

from langgraph.graph import MessagesState, END
from langgraph.types import Command


members = ["market_research", "stock_research", "company_research"]
options = members + ["FINISH"]

system_prompt = (
    "You are a supervisor tasked with managing a conversation between the"
    f" following workers: {members}. Given the following user request,"
    " respond with the worker to act next. Each worker will perform a"
    " task and respond with their results and status. When finished,"
    " respond with FINISH."
)

class Router(TypedDict):
    """Worker to route to next. If no workers needed, route to FINISH."""

    next: Literal[*options]


def supervisor_node(state: MessagesState) -> Command[Literal[*members, "analyst"]]:
    """
    supervisor node입니다. 주어진 State를 기반으로 각 worker의 결과를 종합하고,
    다음에 수행할 worker를 결정합니다. 모든 작업이 완료되면 analyst node로 이동합니다.

    Args:
        state (MessagesState): 현재 메시지 상태를 나타내는 객체입니다.

    Returns:
        Command: 다음에 수행할 worker 또는 analyst node로 이동하기 위한 명령을 반환합니다.
    """
    messages = [
        {"role": "system", "content": system_prompt},
    ] + state["messages"]
    response = llm.with_structured_output(Router).invoke(messages)
    goto = response["next"]
    if goto == "FINISH":
        goto = "analyst"

    return Command(goto=goto)


In [ ]:

from langgraph.graph import StateGraph, START

graph_builder = StateGraph(MessagesState)

graph_builder.add_node("supervisor", supervisor_node)
graph_builder.add_node("market_research", market_research_node)
graph_builder.add_node("stock_research", stock_research_node)
graph_builder.add_node("company_research", company_research_node)
graph_builder.add_node("analyst", analyst_node)

In [ ]:

graph_builder.add_edge(START, "supervisor")
graph_builder.add_edge("analyst", END)
graph = graph_builder.compile()

In [ ]:

from IPython.display import Image, display

display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
for chunk in graph.stream(
    {"messages": [("user", "Would you invest in Snowflake?")]}, stream_mode="values"
):
    chunk['messages'][-1].pretty_print()